# Combat transitions → feature table (Colab tip #1)

**Track:** Colab GPU parallel — feature extract only. Does **not** train Critic/DT, does **not** touch game comms / EP bridge / `sts2.dll`.

**Input:** `transitions.npz` with keys `obs`, `next_obs`, `action`, `reward`, `done`, `action_mask`.

| Field | Shape / dtype |
|-------|----------------|
| `obs`, `next_obs` | `(N, 181)` float32 — combat obs_v1 |
| `action` | `(N,)` int — gym combat action index |
| `reward` | `(N,)` float32 |
| `done` | `(N,)` bool |
| `action_mask` | `(N, M)` — legal action mask (M ≈ action space) |

**Gate:** export ≥1000 rows; **no NaN/Inf** in numeric columns.

**Optional (not gated here):** HOLD replay JSONL `hold_turn_replay_*.jsonl` for hp/enemies_hp context in later tips.

**Outputs (default):** persist under Google Drive `MyDrive/sts2/colab/` (Pharma convention). Local / no Drive: set `USE_DRIVE=False` and use `/content/...` fallback paths.

In [ ]:
# Run once: clone repo for extract logic (or upload sts2_env/colab/ and fix sys.path)
# !git clone --depth 1 https://github.com/EienteiPharma/sts2-rl-agent.git /content/sts2-rl-agent

!pip install -q numpy pandas pyarrow

import sys
from pathlib import Path

REPO_ROOT = Path("/content/sts2-rl-agent")
if not REPO_ROOT.is_dir():
    raise SystemExit(
        "Clone sts2-rl-agent to /content/sts2-rl-agent (uncomment git clone) "
        "or upload combat_transition_features.py and add parent to sys.path."
    )
sys.path.insert(0, str(REPO_ROOT))

from sts2_env.colab.combat_transition_features import extract_combat_features, COMBAT_OBS_DIM

print("COMBAT_OBS_DIM", COMBAT_OBS_DIM)

In [ ]:
# --- Paths (no /workspace); default artifacts on Drive ---
from google.colab import drive, files  # type: ignore

USE_DRIVE = True  # False for local/smoke: writes under /content/sts2/colab/ only
DRIVE_COLAB_DIR = "/content/drive/MyDrive/sts2/colab"
LOCAL_COLAB_DIR = "/content/sts2/colab"

USE_UPLOAD = True  # upload source npz; set False if NPZ_PATH already on Drive/disk
NPZ_PATH = "/content/transitions.npz"  # staging upload (override to Drive path if pref)
OUT_PATH = f"{DRIVE_COLAB_DIR}/combat_features.parquet"  # or .npz — change extension if needed
MIN_ROWS = 1000
SAMPLE_ROWS = 1000
SEED = 0

if USE_DRIVE:
    drive.mount("/content/drive")
    out_dir = DRIVE_COLAB_DIR
else:
    out_dir = LOCAL_COLAB_DIR
    OUT_PATH = f"{LOCAL_COLAB_DIR}/combat_features.parquet"

Path = __import__("pathlib").Path
Path(out_dir).mkdir(parents=True, exist_ok=True)
if USE_DRIVE and not str(OUT_PATH).startswith(DRIVE_COLAB_DIR):
    OUT_PATH = str(Path(DRIVE_COLAB_DIR) / Path(OUT_PATH).name)

if USE_UPLOAD:
    uploaded = files.upload()  # pick transitions.npz in browser
    assert uploaded, "upload transitions.npz"
    name = next(iter(uploaded))
    with open(NPZ_PATH, "wb") as f:
        f.write(uploaded[name])
    print("wrote", NPZ_PATH, "from", name)

print("OUT_PATH", OUT_PATH)

In [ ]:
meta = extract_combat_features(
    NPZ_PATH,
    OUT_PATH,
    min_rows=MIN_ROWS,
    sample_rows=SAMPLE_ROWS,
    seed=SEED,
)
meta

In [ ]:
import numpy as np

assert meta["n_rows"] >= MIN_ROWS
assert meta["obs_shape"][1] == 181
print("OK gate: rows", meta["n_rows"], "obs_shape", meta["obs_shape"])

if OUT_PATH.endswith(".npz"):
    with np.load(OUT_PATH) as z:
        for k in z.files:
            a = z[k]
            if np.issubdtype(a.dtype, np.floating):
                assert np.isfinite(a).all(), k
else:
    import pandas as pd
    df = pd.read_parquet(OUT_PATH)
    num = df.select_dtypes(include=["float", "float32", "float64"])
    assert num.isna().sum().sum() == 0
    print("parquet columns", len(df.columns), "rows", len(df))
    df.head()

## Optional: download to laptop

Default persistence is **Google Drive** (`OUT_PATH` above). Run the next cell only if you want a local copy without Drive sync.

In [ ]:
from google.colab import files  # type: ignore
files.download(OUT_PATH)